In [2]:
%pip install tensorflow

  Using cached tensorflow-2.21.0-cp311-cp311-win_amd64.whl (350.8 MB)
  Using cached absl_py-2.4.0-py3-none-any.whl (135 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl (12 kB)
  Using cached requests-2.33.1-py3-none-any.whl (64 kB)
  Using cached keras-3.14.0-py3-none-any.whl (1.6 MB)
  Using cached rich-15.0.0-py3-none-any.whl (310 kB)
  Using cached certifi-2026.2.25-py3-none-any.whl (153 kB)
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: [WinError 2] The system cannot find the file specified: 'c:\\Python311\\Scripts\\saved_model_cli.exe' -> 'c:\\Python311\\Scripts\\saved_model_cli.exe.deleteme'


[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import tensorflow as tf

In [4]:
import pandas as pd

In [5]:
df = pd.read_csv('all_tickets_processed_improved_v3.csv')

In [6]:
df.head()

,Document,Topic_group
0,connection with icon icon dear please setup ic...,Hardware
1,work experience user work experience user hi w...,Access
2,requesting for meeting requesting meeting hi p...,Hardware
3,reset passwords for external accounts re expir...,Access
4,mail verification warning hi has got attached ...,Miscellaneous


In [7]:
print(df.columns)

Index(['Document', 'Topic_group'], dtype='str')


In [8]:
df.dropna()

,Document,Topic_group
0,connection with icon icon dear please setup ic...,Hardware
1,work experience user work experience user hi w...,Access
2,requesting for meeting requesting meeting hi p...,Hardware
3,reset passwords for external accounts re expir...,Access
4,mail verification warning hi has got attached ...,Miscellaneous
...,...,...
47832,git space for a project issues with adding use...,Access
47833,error sent july error hi guys can you help out...,Miscellaneous
47834,connection issues sent tuesday july connection...,Hardware
47835,error cube reports sent tuesday july error hel...,HR Support


In [9]:
texts=df["Document"]
labels=df["Topic_group"]

In [10]:
import re
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text) # Remove punctuation/special chars
    return text

In [11]:
df["clean_text"] = df["Document"].apply(clean_text)

texts = df["clean_text"]

In [12]:
from tensorflow.keras.preprocessing.text import Tokenizer


In [13]:
Tokenizer=Tokenizer(num_words=10000, oov_token="<OOV>")
Tokenizer.fit_on_texts(texts)

In [15]:
sequences = Tokenizer.texts_to_sequences(texts)

In [16]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_len = 100

X = pad_sequences(sequences, maxlen=max_len, padding='post')

In [20]:
%pip install scikit-learn

     ---------------------------------------- 8.1/8.1 MB 922.5 kB/s eta 0:00:00
     -------------------------------------- 36.6/36.6 MB 690.6 kB/s eta 0:00:00
  Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(labels)

In [22]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [25]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

In [34]:
rnn_model = Sequential([
    Embedding(input_dim=10000, output_dim=64, input_length=100),
    SimpleRNN(64),
    Dense(32, activation='relu'),
    Dense(8, activation='softmax')
])

c:\Python311\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [35]:
rnn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [36]:
print(df.columns)
print(len(set(y)))
print(X.shape)

Index(['Document', 'Topic_group', 'clean_text'], dtype='str')
8
(47837, 100)


In [39]:
rnn_model.fit(X_train, y_train, epochs=10, batch_size=32)

Epoch 1/10
1196/1196 ━━━━━━━━━━━━━━━━━━━━ 47s 39ms/step - accuracy: 0.3224 - loss: 1.7219
Epoch 2/10
1196/1196 ━━━━━━━━━━━━━━━━━━━━ 48s 40ms/step - accuracy: 0.3255 - loss: 1.7128
Epoch 3/10
1196/1196 ━━━━━━━━━━━━━━━━━━━━ 43s 36ms/step - accuracy: 0.3279 - loss: 1.7054
Epoch 4/10
1196/1196 ━━━━━━━━━━━━━━━━━━━━ 86s 39ms/step - accuracy: 0.3288 - loss: 1.7011
Epoch 5/10
1196/1196 ━━━━━━━━━━━━━━━━━━━━ 46s 38ms/step - accuracy: 0.3291 - loss: 1.7001
Epoch 6/10
1196/1196 ━━━━━━━━━━━━━━━━━━━━ 43s 36ms/step - accuracy: 0.3300 - loss: 1.6962
Epoch 7/10
1196/1196 ━━━━━━━━━━━━━━━━━━━━ 38s 32ms/step - accuracy: 0.3313 - loss: 1.6920
Epoch 8/10
1196/1196 ━━━━━━━━━━━━━━━━━━━━ 36s 30ms/step - accuracy: 0.3307 - loss: 1.6897
Epoch 9/10
1196/1196 ━━━━━━━━━━━━━━━━━━━━ 39s 32ms/step - accuracy: 0.3326 - loss: 1.6887
Epoch 10/10
1196/1196 ━━━━━━━━━━━━━━━━━━━━ 39s 33ms/step - accuracy: 0.3331 - loss: 1.6864


In [40]:
rnn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [41]:
from sklearn.metrics import classification_report

y_pred = rnn_model.predict(X_test)
y_pred = y_pred.argmax(axis=1)

print(classification_report(y_test, y_pred))

299/299 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step
              precision    recall  f1-score   support

           0       0.24      0.02      0.04      1455
           1       0.50      0.09      0.15       342
           2       0.58      0.07      0.12      2107
           3       0.29      0.92      0.44      2760
           4       0.18      0.02      0.03       451
           5       0.30      0.03      0.06      1400
           6       0.09      0.00      0.00       497
           7       0.11      0.01      0.02       556

    accuracy                           0.29      9568
   macro avg       0.28      0.14      0.11      9568
weighted avg       0.33      0.29      0.17      9568



In [43]:
from tensorflow.keras.layers import LSTM

lstm_model = Sequential([
    Embedding(input_dim=10000, output_dim=64, input_length=max_len),
    LSTM(64),
    Dense(32, activation='relu'),
    Dense(8, activation='softmax')
])

In [44]:
lstm_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

lstm_model.fit(X_train, y_train, epochs=5, batch_size=32)

Epoch 1/5
1196/1196 ━━━━━━━━━━━━━━━━━━━━ 96s 77ms/step - accuracy: 0.3120 - loss: 1.7974
Epoch 2/5
1196/1196 ━━━━━━━━━━━━━━━━━━━━ 75s 62ms/step - accuracy: 0.3489 - loss: 1.7439
Epoch 3/5
1196/1196 ━━━━━━━━━━━━━━━━━━━━ 76s 64ms/step - accuracy: 0.4870 - loss: 1.4073
Epoch 4/5
1196/1196 ━━━━━━━━━━━━━━━━━━━━ 79s 66ms/step - accuracy: 0.6535 - loss: 0.9893
Epoch 5/5
1196/1196 ━━━━━━━━━━━━━━━━━━━━ 79s 66ms/step - accuracy: 0.7709 - loss: 0.6823


In [47]:
y_pred_lstm = lstm_model.predict(X_test)
y_pred_lstm = y_pred_lstm.argmax(axis=1)

lstm_accuracy=print(classification_report(y_test, y_pred_lstm))

299/299 ━━━━━━━━━━━━━━━━━━━━ 6s 20ms/step
              precision    recall  f1-score   support

           0       0.79      0.88      0.83      1455
           1       0.63      0.57      0.60       342
           2       0.81      0.85      0.83      2107
           3       0.77      0.86      0.81      2760
           4       0.77      0.59      0.67       451
           5       0.66      0.60      0.63      1400
           6       0.96      0.78      0.86       497
           7       0.92      0.57      0.70       556

    accuracy                           0.78      9568
   macro avg       0.79      0.71      0.74      9568
weighted avg       0.78      0.78      0.77      9568



In [46]:
lstm_model.save("lstm_model.h5")

In [54]:
# the LSTM model is better than the RNN model at least with 2 multoplay accuracy
# with half of epochs thanks to the memory skills of LSTM.
print (f"LSTM Accuracy:\n {classification_report(y_test, y_pred_lstm)} VS.\n RNN Accuracy:\n {classification_report(y_test, y_pred)}")


LSTM Accuracy:
               precision    recall  f1-score   support

           0       0.79      0.88      0.83      1455
           1       0.63      0.57      0.60       342
           2       0.81      0.85      0.83      2107
           3       0.77      0.86      0.81      2760
           4       0.77      0.59      0.67       451
           5       0.66      0.60      0.63      1400
           6       0.96      0.78      0.86       497
           7       0.92      0.57      0.70       556

    accuracy                           0.78      9568
   macro avg       0.79      0.71      0.74      9568
weighted avg       0.78      0.78      0.77      9568
 VS.
 RNN Accuracy:
               precision    recall  f1-score   support

           0       0.24      0.02      0.04      1455
           1       0.50      0.09      0.15       342
           2       0.58      0.07      0.12      2107
           3       0.29      0.92      0.44      2760
           4       0.18      0.02      0.0